In [335]:
import re
import json
import requests
import pandas as pd
from sparql_generate_query import run_minmod_query
import networkx as nx


In [336]:
query = ''' SELECT ?ms1 ?ms2
            WHERE {
                ?ms1 a :MineralSite .
                ?ms2 a :MineralSite .

                ?ms1 owl:sameAs ?ms2 .
            } '''
df_all_sites = run_minmod_query(query, values=True)
df_all_sites

,ms1.value,ms2.value
0,https://minmod.isi.edu/resource/mineral_sitef3...,https://minmod.isi.edu/resource/mineral_site99...
1,https://minmod.isi.edu/resource/site__sudbury-...,https://minmod.isi.edu/resource/mineral_site39...
2,https://minmod.isi.edu/resource/mineral_sitef1...,https://minmod.isi.edu/resource/mineral_site30...
3,https://minmod.isi.edu/resource/mineral_sitea8...,https://minmod.isi.edu/resource/mineral_site2f...
4,https://minmod.isi.edu/resource/mineral_site1a...,https://minmod.isi.edu/resource/mineral_sitea9...
...,...,...
1230,https://minmod.isi.edu/resource/mineral_sitefc...,https://minmod.isi.edu/resource/mineral_site9d...
1231,https://minmod.isi.edu/resource/mineral_site36...,https://minmod.isi.edu/resource/mineral_site69...
1232,https://minmod.isi.edu/resource/site__sudbury-...,https://minmod.isi.edu/resource/mineral_site39...
1233,https://minmod.isi.edu/resource/mineral_site60...,https://minmod.isi.edu/resource/mineral_site9d...


In [337]:
G = nx.from_pandas_edgelist(df_all_sites, source='ms1.value', target='ms2.value')

# Find connected components
groups = nx.connected_components(G)

# Create a mapping from value to group ID
group_mapping = {}
for group_id, group in enumerate(groups, start=1):
    for value in group:
        group_mapping[value] = group_id

# Map group IDs to the dataframe
df_all_sites['group_id'] = df_all_sites['ms1.value'].map(group_mapping)

(df_all_sites)

,ms1.value,ms2.value,group_id
0,https://minmod.isi.edu/resource/mineral_sitef3...,https://minmod.isi.edu/resource/mineral_site99...,1
1,https://minmod.isi.edu/resource/site__sudbury-...,https://minmod.isi.edu/resource/mineral_site39...,2
2,https://minmod.isi.edu/resource/mineral_sitef1...,https://minmod.isi.edu/resource/mineral_site30...,3
3,https://minmod.isi.edu/resource/mineral_sitea8...,https://minmod.isi.edu/resource/mineral_site2f...,4
4,https://minmod.isi.edu/resource/mineral_site1a...,https://minmod.isi.edu/resource/mineral_sitea9...,5
...,...,...,...
1230,https://minmod.isi.edu/resource/mineral_sitefc...,https://minmod.isi.edu/resource/mineral_site9d...,379
1231,https://minmod.isi.edu/resource/mineral_site36...,https://minmod.isi.edu/resource/mineral_site69...,53
1232,https://minmod.isi.edu/resource/site__sudbury-...,https://minmod.isi.edu/resource/mineral_site39...,2
1233,https://minmod.isi.edu/resource/mineral_site60...,https://minmod.isi.edu/resource/mineral_site9d...,19


In [338]:
df_all_sites.to_csv('/Users/namratasharma/Documents/USC/OnCampus/DARPA/ta2-minmod-data/sandbox/data/sameas_mineralsites_group_id.csv', index=False, mode='w')


In [339]:
# Query to get all deposit types and their confidence for Mineral Sites

query = '''
SELECT 
  ?ms                                              # Mineral Site URI
  ?ms_name                                         # Mineral Site Name
  ?deposit_name                                    # Deposit Type Name
  ?deposit_confidence                                    # Deposit Type Confidence
  ?deposit_source
  ?deposit_group
  ?deposit_environment
  ?country                                         # Country
  ?loc_wkt                                         # WKT Geometry
  ?state_or_province
  ?total_tonnage_measured
  ?total_tonnage_indicated
  ?total_tonnage_inferred
  ?total_contained_measured
  ?total_contained_indicated
  ?total_contained_inferred
  (?total_tonnage_measured + ?total_tonnage_indicated + ?total_tonnage_inferred AS ?total_tonnage)                     # Total Tonnage [million tonnes]
  (?total_contained_measured + ?total_contained_indicated + ?total_contained_inferred AS ?total_contained_metal)
  (IF(?total_tonnage > 0, ?total_contained_metal / ?total_tonnage, 0) AS ?total_grade)                                 # Total Grade
WHERE {
  {
    SELECT ?ms ?ms_name ?deposit_name ?deposit_confidence ?country ?loc_wkt ?state_or_province ?deposit_source ?deposit_group ?deposit_environment
           (SUM(?tonnage_measured) AS ?total_tonnage_measured)
           (SUM(?tonnage_indicated) AS ?total_tonnage_indicated)
           (SUM(?tonnage_inferred) AS ?total_tonnage_inferred)
           (SUM(?contained_measured) AS ?total_contained_measured)
           (SUM(?contained_indicated) AS ?total_contained_indicated)
           (SUM(?contained_inferred) AS ?total_contained_inferred)
    WHERE {
    
        ?ms :deposit_type_candidate ?deposit_candidate_uri .
        ?deposit_candidate_uri :source ?deposit_source .
        ?deposit_candidate_uri :confidence ?deposit_confidence .
        ?deposit_candidate_uri :normalized_uri [ rdfs:label ?deposit_name ] .
        ?deposit_candidate_uri :normalized_uri [ :deposit_group ?deposit_group ] .
        ?deposit_candidate_uri :normalized_uri [ :environment ?deposit_environment ] .
    
        ?ms :mineral_inventory ?mi .
        OPTIONAL { ?ms rdfs:label|:name ?ms_name . }
        
        ?ms :location_info ?loc .
        
        OPTIONAL { ?loc :country ?country . }
        OPTIONAL { ?loc :state_or_province ?state_or_province . }
        OPTIONAL { ?loc :location ?loc_wkt . }
        #FILTER(datatype(?loc_wkt) = geo:wktLiteral)
        
        ?mi :category ?mi_cat .
        ?mi :commodity [ :name "Nickel"@en ] .
    
        {
            SELECT ?mi (MAX(?ore_val) AS ?max_ore_val) (SAMPLE(?grade_val) AS ?matched_grade_val)
            WHERE {
                ?mi :ore [ :ore_value ?ore_val_raw; :ore_unit ?ore_unit] .
                OPTIONAL { ?mi :grade [ :grade_value ?grade_val; :grade_unit ?grade_unit] . }
                BIND(IF(bound(?ore_val_raw), ?ore_val_raw, 0) AS ?ore_val_pre)
                BIND(IF(?ore_unit = <https://minmod.isi.edu/resource/Q202>, ?ore_val_pre, IF(?ore_unit = <https://minmod.isi.edu/resource/Q200>, ?ore_val_pre / 1e6, ?ore_val_pre)) AS ?ore_val)
            }
            GROUP BY ?mi
        }
    
        BIND(IF(CONTAINS(LCASE(STR(?mi_cat)), "measured"), ?max_ore_val, 0) AS ?tonnage_measured)
        BIND(IF(CONTAINS(LCASE(STR(?mi_cat)), "indicated"), ?max_ore_val, 0) AS ?tonnage_indicated)
        BIND(IF(CONTAINS(LCASE(STR(?mi_cat)), "inferred"), ?max_ore_val, 0) AS ?tonnage_inferred)
    
        BIND(IF(CONTAINS(LCASE(STR(?mi_cat)), "measured") && ?matched_grade_val > 0, ?max_ore_val * ?matched_grade_val, 0) AS ?contained_measured)
        BIND(IF(CONTAINS(LCASE(STR(?mi_cat)), "indicated") && ?matched_grade_val > 0, ?max_ore_val * ?matched_grade_val, 0) AS ?contained_indicated)
        BIND(IF(CONTAINS(LCASE(STR(?mi_cat)), "inferred") && ?matched_grade_val > 0, ?max_ore_val * ?matched_grade_val, 0) AS ?contained_inferred)
    
    }
    GROUP BY ?ms ?ms_name ?deposit_name ?deposit_confidence ?country ?loc_wkt ?state_or_province ?deposit_group ?deposit_source ?deposit_environment
  }
}
'''
query_resp_df = run_minmod_query(query, values=True)
# print(query_resp_df)
print(f'queried & aggregated data from {len(query_resp_df)} sites...')
# print(query_resp_df)
gt_data_df = pd.DataFrame([
    {
        'ms': row['ms.value'],
        'ms_name': row['ms_name.value'] if len(row['ms_name.value']) > 0 else row['ms.value'].split('/')[-1],
        'deposit_type': row['deposit_name.value'],
        'deposit_type_confidence': row['deposit_confidence.value'],
        'deposit_source': row['deposit_source.value'],
        'deposit_group': row['deposit_group.value'],
        'deposit_environment': row['deposit_environment.value'],
        'country': row['country.value'],
        'state_or_province': row['state_or_province.value'],
        'loc_wkt': row['loc_wkt.value'],
        'total_tonnage':float(row['total_tonnage.value']),
        'total_grade': float(row['total_grade.value'])
    }
    for index, row in query_resp_df.iterrows()
])

queried & aggregated data from 269 sites...


In [340]:
gt_data_df

,ms,ms_name,deposit_type,deposit_type_confidence,deposit_source,deposit_group,deposit_environment,country,state_or_province,loc_wkt,total_tonnage,total_grade
0,https://minmod.isi.edu/resource/site__snipe-ba...,Snipe Bay,U-M layered intrusion nickel- copper-PGE,1e+00,expert,Ultramafic and (or) mafic-layered intrusion,Magmatic,USA,NaN,POINT (-134.95753 56.42213),0.780000,0.300000
1,https://minmod.isi.edu/resource/mineral_sitedb...,Technical Report on the Preliminary Assessment...,Arc U-M intru- sion nickel- copper-PGE,0.07385503500699997,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic,USA,Minnesota,POINT(-91.8 47.733333),167.970430,0.239971
2,https://minmod.isi.edu/resource/site__local-bo...,Local Boy,U-M layered intrusion nickel- copper-PGE,1e+00,expert,Ultramafic and (or) mafic-layered intrusion,Magmatic,USA,NaN,NaN,9.080000,0.360000
3,https://minmod.isi.edu/resource/mineral_site5f...,NI 43-101 Technical Report on the Maturi Birch...,U-M intrusion nickel-copper- PGE,0.10459203273057938,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic,USA,Minnesota,"MULTIPOINT(-91.70833 47.78333,-91.79167 47.696...",5321.000000,0.182704
4,https://minmod.isi.edu/resource/mineral_sitece...,NI 43-101 Technical Report - Ban Phuc Nickel P...,U-M layered intrusion nickel- copper-PGE,0.06899438798427582,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic-layered intrusion,Magmatic,NaN,NaN,NaN,8.340000,2.108225
...,...,...,...,...,...,...,...,...,...,...,...,...
264,https://minmod.isi.edu/resource/mineral_site79...,NI 43-101 Technical Report for the Alexo Proje...,Arc U-M intru- sion nickel- copper-PGE,0.04167810454964638,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic,Canada,Ontario,POINT(-80.811947 48.658097),0.981001,1.102681
265,https://minmod.isi.edu/resource/mineral_site5d...,NI 43-101 Technical Report for the Thunder Bay...,U-M layered intrusion nickel- copper-PGE,0.0836910530924797,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic-layered intrusion,Magmatic,NaN,NaN,NaN,1.083000,0.237064
266,https://minmod.isi.edu/resource/mineral_site31...,NI 43-101 Technical Report for the Crawford Pr...,Vein ± replacement nickel,0.059374239295721054,"algorithm predictions, SRI deposit type classi...",Vein,Magmatic hydrothermal,Canada,Ontario,POINT(-81.3335 48.8073),3622.900000,0.248203
267,https://minmod.isi.edu/resource/mineral_site03...,NI 43-101 Technical Report for the Nickel Roya...,MVT zinc-lead,0.049673642963171005,"algorithm predictions, SRI deposit type classi...",Mississippi Valley- type (MVT),Basin hydrothermal,USA,Minnesota,NaN,0.000000,0.000000


In [341]:
gt_data_df.to_csv('gt_model_group_query_results.csv', index=False, mode='w')

In [342]:
df_melted = df_all_sites.melt(id_vars=['group_id'], value_vars=['ms1.value', 'ms2.value'], value_name='ms')

df_all_sites_groups = df_melted[['ms', 'group_id']].drop_duplicates()

df_all_sites_groups

,ms,group_id
0,https://minmod.isi.edu/resource/mineral_sitef3...,1
1,https://minmod.isi.edu/resource/site__sudbury-...,2
2,https://minmod.isi.edu/resource/mineral_sitef1...,3
3,https://minmod.isi.edu/resource/mineral_sitea8...,4
4,https://minmod.isi.edu/resource/mineral_site1a...,5
...,...,...
1230,https://minmod.isi.edu/resource/mineral_sitefc...,379
1231,https://minmod.isi.edu/resource/mineral_site36...,53
1232,https://minmod.isi.edu/resource/site__sudbury-...,2
1233,https://minmod.isi.edu/resource/mineral_site60...,19


In [343]:
merged_df_all_sites_all_dep = pd.merge(gt_data_df, df_all_sites_groups, how='left', on='ms')
merged_df_all_sites_all_dep

,ms,ms_name,deposit_type,deposit_type_confidence,deposit_source,deposit_group,deposit_environment,country,state_or_province,loc_wkt,total_tonnage,total_grade,group_id
0,https://minmod.isi.edu/resource/site__snipe-ba...,Snipe Bay,U-M layered intrusion nickel- copper-PGE,1e+00,expert,Ultramafic and (or) mafic-layered intrusion,Magmatic,USA,NaN,POINT (-134.95753 56.42213),0.780000,0.300000,6.0
1,https://minmod.isi.edu/resource/mineral_sitedb...,Technical Report on the Preliminary Assessment...,Arc U-M intru- sion nickel- copper-PGE,0.07385503500699997,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic,USA,Minnesota,POINT(-91.8 47.733333),167.970430,0.239971,NaN
2,https://minmod.isi.edu/resource/site__local-bo...,Local Boy,U-M layered intrusion nickel- copper-PGE,1e+00,expert,Ultramafic and (or) mafic-layered intrusion,Magmatic,USA,NaN,NaN,9.080000,0.360000,NaN
3,https://minmod.isi.edu/resource/mineral_site5f...,NI 43-101 Technical Report on the Maturi Birch...,U-M intrusion nickel-copper- PGE,0.10459203273057938,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic,USA,Minnesota,"MULTIPOINT(-91.70833 47.78333,-91.79167 47.696...",5321.000000,0.182704,61.0
4,https://minmod.isi.edu/resource/mineral_sitece...,NI 43-101 Technical Report - Ban Phuc Nickel P...,U-M layered intrusion nickel- copper-PGE,0.06899438798427582,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic-layered intrusion,Magmatic,NaN,NaN,NaN,8.340000,2.108225,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,https://minmod.isi.edu/resource/mineral_site79...,NI 43-101 Technical Report for the Alexo Proje...,Arc U-M intru- sion nickel- copper-PGE,0.04167810454964638,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic,Canada,Ontario,POINT(-80.811947 48.658097),0.981001,1.102681,NaN
265,https://minmod.isi.edu/resource/mineral_site5d...,NI 43-101 Technical Report for the Thunder Bay...,U-M layered intrusion nickel- copper-PGE,0.0836910530924797,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic-layered intrusion,Magmatic,NaN,NaN,NaN,1.083000,0.237064,NaN
266,https://minmod.isi.edu/resource/mineral_site31...,NI 43-101 Technical Report for the Crawford Pr...,Vein ± replacement nickel,0.059374239295721054,"algorithm predictions, SRI deposit type classi...",Vein,Magmatic hydrothermal,Canada,Ontario,POINT(-81.3335 48.8073),3622.900000,0.248203,NaN
267,https://minmod.isi.edu/resource/mineral_site03...,NI 43-101 Technical Report for the Nickel Roya...,MVT zinc-lead,0.049673642963171005,"algorithm predictions, SRI deposit type classi...",Mississippi Valley- type (MVT),Basin hydrothermal,USA,Minnesota,NaN,0.000000,0.000000,NaN


In [344]:
merged_df_all_sites_all_dep.to_csv('gt_model_group_id.csv', index=False, mode='w')

In [345]:
max_group_id = merged_df_all_sites_all_dep['group_id'].fillna(0).max()
merged_df_all_sites_all_dep['group_id'] = merged_df_all_sites_all_dep['group_id'].fillna(pd.Series(range(int(max_group_id) + 1, len(merged_df_all_sites_all_dep) + int(max_group_id) + 1)))
sorted_df_all_sites_all_dep = merged_df_all_sites_all_dep.sort_values(by=['group_id', 'ms_name', 'deposit_type_confidence'])


In [346]:
sorted_df_all_sites_all_dep.to_csv('gt_model_group_id2.csv', index=False, mode='w')


In [347]:
# The Grade Tonnage Dataframe

gt_df_only = sorted_df_all_sites_all_dep[['ms', 'ms_name', 'country', 'state_or_province', 'loc_wkt', 'total_tonnage', 'total_grade']]

gt_df_only = gt_df_only.drop_duplicates(subset=['ms', 'ms_name', 'country', 'state_or_province', 'loc_wkt', 'total_tonnage', 'total_grade'])

# Reset index if needed
newgt_df_only_df = gt_df_only.reset_index(drop=True)

In [348]:
newgt_df_only_df

,ms,ms_name,country,state_or_province,loc_wkt,total_tonnage,total_grade
0,https://minmod.isi.edu/resource/site__snipe-ba...,Snipe Bay,USA,NaN,POINT (-134.95753 56.42213),0.780000,0.300000
1,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,NaN,NaN,NaN,24.806000,1.651954
2,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,United States,Minnesota,POINT(-93.10603 46.697),12.403000,1.651954
3,https://minmod.isi.edu/resource/mineral_site5f...,NI 43-101 Technical Report on the Maturi Birch...,USA,Minnesota,"MULTIPOINT(-91.70833 47.78333,-91.79167 47.696...",5321.000000,0.182704
4,https://minmod.isi.edu/resource/mineral_site5f...,NI 43-101 Technical Report on the Maturi Birch...,NaN,NaN,NaN,5321.000000,0.182704
...,...,...,...,...,...,...,...
62,https://minmod.isi.edu/resource/mineral_sitef4...,NI 43-101 Technical Report Regarding Update to...,NaN,NaN,NaN,25.917000,1.363711
63,https://minmod.isi.edu/resource/mineral_site79...,NI 43-101 Technical Report for the Alexo Proje...,NaN,NaN,NaN,1.962002,1.102681
64,https://minmod.isi.edu/resource/mineral_site79...,NI 43-101 Technical Report for the Alexo Proje...,Canada,Ontario,POINT(-80.811947 48.658097),1.962002,1.102681
65,https://minmod.isi.edu/resource/site__spruce-r...,Spruce Road,USA,NaN,POINT (-91.7 47.83),871.000000,0.160000


In [349]:
duplicates = newgt_df_only_df.duplicated(subset=['ms'], keep=False)

if duplicates.any():
    newgt_df_only_df = newgt_df_only_df.dropna(subset=['loc_wkt'])
    
newgt_df_only_df

,ms,ms_name,country,state_or_province,loc_wkt,total_tonnage,total_grade
0,https://minmod.isi.edu/resource/site__snipe-ba...,Snipe Bay,USA,NaN,POINT (-134.95753 56.42213),0.780000,0.300000
2,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,United States,Minnesota,POINT(-93.10603 46.697),12.403000,1.651954
3,https://minmod.isi.edu/resource/mineral_site5f...,NI 43-101 Technical Report on the Maturi Birch...,USA,Minnesota,"MULTIPOINT(-91.70833 47.78333,-91.79167 47.696...",5321.000000,0.182704
6,https://minmod.isi.edu/resource/mineral_site2e...,First Independent Technical Report on the Tama...,United States,Minnesota,POINT(-93.1 46.683),9.989000,1.441413
7,https://minmod.isi.edu/resource/site__bohemia-...,Bohemia Basin (Basin-Takanis-Flapjack),USA,NaN,POINT (-136.43427 57.98471),40.000000,0.420000
8,https://minmod.isi.edu/resource/site__northmet...,NorthMet (Dunka Road),USA,NaN,POINT (-92 47.6),2316.600000,0.072048
9,https://minmod.isi.edu/resource/site__mesaba-b...,Mesaba (Babbit),USA,NaN,POINT (-91.9 47.63),2214.000000,0.140000
10,https://minmod.isi.edu/resource/site__maturi-s...,Maturi Southwest,USA,NaN,POINT (-91.8 47.8),244.800000,0.165212
11,https://minmod.isi.edu/resource/site__maturi-i...,Maturi (incl. Nokomis),USA,NaN,POINT (-91.8 47.8),3287.600000,0.177047
12,https://minmod.isi.edu/resource/mineral_site98...,Preliminary Economic Assessment (PEA) #3 of th...,United States,Minnesota,MULTIPOINT(-92.7 46.7),22.178000,1.395021


In [350]:
newgt_df_only_df.to_csv('gt_model_mineral_sites.csv', index=False, mode='w')


In [351]:
# Mineral Site to Deposit Type
gt_dt_only = sorted_df_all_sites_all_dep[['ms', 'ms_name', 'country', 'state_or_province', 'loc_wkt', 'deposit_type', 'deposit_type_confidence', 'deposit_source', 'deposit_group', 'deposit_environment']]

gt_dt_only = gt_dt_only.drop_duplicates(subset= ['ms', 'ms_name', 'country', 'state_or_province', 'loc_wkt', 'deposit_type', 'deposit_type_confidence', 'deposit_source', 'deposit_group', 'deposit_environment'])

# Reset index if needed
newgt_dt_only_df = gt_dt_only.reset_index(drop=True)

In [352]:
newgt_dt_only_df

,ms,ms_name,country,state_or_province,loc_wkt,deposit_type,deposit_type_confidence,deposit_source,deposit_group,deposit_environment
0,https://minmod.isi.edu/resource/site__snipe-ba...,Snipe Bay,USA,NaN,POINT (-134.95753 56.42213),U-M layered intrusion nickel- copper-PGE,1e+00,expert,Ultramafic and (or) mafic-layered intrusion,Magmatic
1,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,NaN,NaN,NaN,U-M conduit nickel-copper- PGE,0.04616041108965874,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic conduit,Magmatic
2,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,United States,Minnesota,POINT(-93.10603 46.697),U-M conduit nickel-copper- PGE,0.04616041108965874,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic conduit,Magmatic
3,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,NaN,NaN,NaN,U-M conduit nickel-copper- PGE,0.04663281887769699,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic conduit,Magmatic
4,https://minmod.isi.edu/resource/mineral_site6a...,Preliminary Economic Assessment (PEA) of the T...,United States,Minnesota,POINT(-93.10603 46.697),U-M conduit nickel-copper- PGE,0.04663281887769699,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic conduit,Magmatic
...,...,...,...,...,...,...,...,...,...,...
264,https://minmod.isi.edu/resource/mineral_site79...,NI 43-101 Technical Report for the Alexo Proje...,Canada,Ontario,POINT(-80.811947 48.658097),Arc U-M intru- sion nickel- copper-PGE,0.04167810454964638,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic intrusion,Magmatic
265,https://minmod.isi.edu/resource/mineral_site5d...,NI 43-101 Technical Report for the Thunder Bay...,NaN,NaN,NaN,U-M layered intrusion nickel- copper-PGE,0.0836910530924797,"algorithm predictions, SRI deposit type classi...",Ultramafic and (or) mafic-layered intrusion,Magmatic
266,https://minmod.isi.edu/resource/mineral_site31...,NI 43-101 Technical Report for the Crawford Pr...,Canada,Ontario,POINT(-81.3335 48.8073),Vein ± replacement nickel,0.059374239295721054,"algorithm predictions, SRI deposit type classi...",Vein,Magmatic hydrothermal
267,https://minmod.isi.edu/resource/mineral_site03...,NI 43-101 Technical Report for the Nickel Roya...,USA,Minnesota,NaN,MVT zinc-lead,0.049673642963171005,"algorithm predictions, SRI deposit type classi...",Mississippi Valley- type (MVT),Basin hydrothermal


In [ ]:
duplicates = newgt_dt_only_df.duplicated(subset=['ms'], keep=False)

# Drop rows where both 'grade' and 'tonnage' are empty if 'ms' has duplicates
if duplicates.any():
    newgt_dt_only_df = newgt_dt_only_df.drop(newgt_dt_only_df[(newgt_dt_only_df['country'] == '') & (newgt_dt_only_df['state_or_province'] == '') & (newgt_dt_only_df['loc_wkt'] == '')].index)
    
    

In [353]:
newgt_dt_only_df.to_csv('gt_model_mineral_sites_deposit_types.csv', index=False, mode='w')


In [354]:
# Hyper Site to Mineral Site
gt_hypersite_only = sorted_df_all_sites_all_dep[['ms', 'group_id', 'ms_name', 'country', 'state_or_province', 'loc_wkt']]

gt_hypersite_only = gt_hypersite_only.drop_duplicates(subset= ['ms', 'group_id', 'ms_name', 'country', 'state_or_province', 'loc_wkt'])

# Reset index if needed
gt_hypersite_only = gt_hypersite_only.reset_index(drop=True)

In [ ]:
duplicates = newgt_df_only_df.duplicated(subset=['ms'], keep=False)

if duplicates.any():
    newgt_df_only_df = newgt_df_only_df.dropna(subset=['loc_wkt'])
    
newgt_df_only_df

In [355]:
gt_hypersite_only.to_csv('gt_model_mineral_sites_hypersites.csv', index=False, mode='w')
